<a href="https://colab.research.google.com/github/PunamBhatia/LogicMojo-AI-ML-April26-Punam/blob/main/MLP_Assignments/AnimalImageClassification_ComputerVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch, torchvision

print("torch version",torch.__version__)
print("torch vision version", torchvision.__version__)
print("Is cuda available", torch.cuda.is_available())

torch version 2.11.0+cpu
torch vision version 0.26.0+cpu
Is cuda available False


In [ ]:
!pip install -q datasets scikit-learn

In [ ]:
import os
#built-in Python module to interact with operating system directly
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models



SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

DEVICE= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device: ",DEVICE)


In [ ]:
#Let's load dataset
#dataset_id="AlvaroVasquezAI/Animal_Image_Classification_Dataset"
dataset_id="microsoft/cats_vs_dogs"
from datasets import load_dataset
from datasets import Features, ClassLabel, Value

# # Define a generic schema so it doesn't try to parse ClassLabels on download
# custom_features = Features({
#     "image": Value("string"),  # or Image() if it's an image path column
#     "label": Value("string")   # Force label to be read as plain text/strings
# })

raw=load_dataset(dataset_id)
print(raw)

In [ ]:
#lets split train, test, val

split_1=raw['train'].train_test_split(test_size=0.3, seed=SEED, stratify_by_column="labels")
split_2=split_1['test'].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="labels")

train_dataset=split_1['train']
test_dataset=split_2['train']
val_dataset=split_2['test']

label_names=raw['train'].features['labels'].names

print(raw)
print(label_names)
print("train, test and val counts", len(train_dataset), len(test_dataset), len(val_dataset))

In [ ]:
#lets view data for EDA

fig,axes= plt.subplots(nrows=3, ncols=4, figsize=(12,9))
idxs=random.sample(range(len(train_dataset)),12)

for ax, i in zip(axes.flat, idxs):
  ex=train_dataset[i]
  ax.imshow(ex["image"])
  ax.set_title(label_names[(ex["labels"])])

plt.suptitle("Random training sampels")
plt.tight_layout()
plt.show()

In [ ]:
#lets check the distribution of data
counts= np.bincount([train_dataset[i]["labels"] for i in range(len(train_dataset))])
for name,c in zip(label_names, counts):
    print(f"{name}:{c} images")

In [ ]:
##Lets resize the images , apply few augmentations and normalize
#for normalization we use mean and std  from existing dataset
IMG_SIZE=244
IMAGB_MEAN=[0.485, 0.456, 0.406]
IMAGB_STD=[0.229, 0.224, 0.225]
train_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGB_MEAN, IMAGB_STD)
])


eval_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGB_MEAN, IMAGB_STD)
])




In [ ]:
#since we have custom data set of hugging face we have to make it compatible to pytorch
class HFAnimalDataset(Dataset):
  def __init__(self, hf_dataset, transform):
    self.ids=hf_dataset
    self.transform=transform

  def __len__(self):
   return len(self.ids)

  def __getitem__(self, idx):
      ex=self.ids[idx]
      image=ex["image"].convert("RGB")
      return self.transform(image), ex["labels"]

train_dataset=HFAnimalDataset(train_dataset, train_transform)
test_dataset=HFAnimalDataset(test_dataset, eval_transform)
val_dataset=HFAnimalDataset(val_dataset, eval_transform)

BATCH_SIZE=32
train_loader=DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader=DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Val batches: {len(val_loader)}")


In [ ]:
#Now we will start with transfer learning
#1)  freez all layers+ add new head
#2) Unfreeze last layer+ new classifier head
#3) Unfreeze last 3-4 layers+ new classifier head (stop before overfitting)
#4) full fine tunning

**Model**

In [ ]:
NUM_CLASSES= len(label_names)
print(NUM_CLASSES)
def build_model(num_classes=NUM_CLASSES):
    # Load ResNet18 with weights learned on ImageNet.
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze all existing layers: keep these features exactly as they are.
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final layer with a fresh one for our 3 classes.
    # Read in_features from the old layer instead of hardcoding 512.
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)  # new layer is trainable by default

    return model


#

In [ ]:
#run model
model=build_model().to(DEVICE)

In [ ]:
#calculate trainable params
#print(len(model.parameters()))
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params=sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params}")
print(f"Trainable params: {trainable_params}")
#

In [ ]:
#lets write one epoch
def run_epoch(model, loader, criterion, optimizer=None):
  is_train=optimizer is not None
  if optimizer is not None:
      model.train()
  else:
      model.eval()

  total_loss,correct,n=0.0, 0 ,0
  torch.set_grad_enabled(is_train)
  for images,labels in loader:
      images,labels=images.to(DEVICE), labels.to(DEVICE)

      if(is_train):
          optimizer.zero_grad()

      outputs=model(images)
      loss=criterion(outputs, labels)
      if(is_train):
          loss.backward()
          optimizer.step()

      total_loss+=loss.item() * images.size(0)
      # _,predicted=torch.max(outputs,1)
      correct+=(outputs.argmax(1) ==labels).sum().item()
      n += images.size(0)
      n+=len(labels)
  torch.set_grad_enabled(True)
  return total_loss / n, correct / n

#train for each epoch and print train/val loss and accuracy for each epoch
def fit(model, train_loader, val_loader, epochs, lr, params=None, phase_name="phase"):
  criterion= nn.CrossEntropyLoss()
  optimizer= optim.Adam(model.parameters(), lr=lr)

  history={"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
  for epoch in range(epochs+1):
      train_loss, train_acc=run_epoch(model, train_loader, criterion, optimizer)
      val_loss, val_acc=run_epoch(model, val_loader, criterion,optimizer=None)
      history["train_loss"].append(train_loss)
      history["val_loss"].append(val_loss)
      history["train_acc"].append(train_acc)
      history["val_acc"].append(val_acc)
      print(f"{phase_name} epoch {epoch}/{epochs} | train loss:{train_loss} train acc:{train_acc} | val loss:{val_loss} val acc:{val_acc}")
  return history


In [ ]:
# Phase 1: train only the new final layer.
history_head = fit(model, train_loader, val_loader, epochs=5, lr=1e-3, phase_name="head-only")


In [ ]:
#lets unfreeze layer4
for param in model.layer4.parameters():
    param.requires_grad = True
trainable= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable}")
total= sum(p.numel() for p in model.parameters())
print(f"Total params: {total}")
## Smaller learning rate (1e-4) and fewer epochs (3): gentle changes to good weights.
finetune_params = [p for p in model.parameters() if p.requires_grad]

history_finetune = fit(model, train_loader, val_loader, epochs=3, lr=1e-4, params=finetune_params, phase_name="finetune layer4")

#

In [ ]:
#lets plot loss and acc

def plot_history(h1, h2, switch_epoch):
  train_loss= h1["train_loss"]+ h2["train_loss"]
  val_loss= h1["val_loss"]+ h2["val_loss"]
  train_acc= h1["train_acc"]+ h2["train_acc"]
  val_acc= h1["val_acc"]+ h2["val_acc"]
  epochs=range(1, len(train_loss)+1)

  fig, axes = plt.subplots(1, 2, figsize=(12, 4))
  for ax,tr, va, title in [
      (axes[0], train_loss, val_loss, "Loss"),
      (axes[1], train_acc, val_acc, "Accuracy")
  ]:
    ax.plot(epochs, tr, label="train",marker="o")
    ax.plot(epochs, va, label="val", marker="o")
    ax.set_title(title)
    ax.axvline(switch_epoch + 0.5, color="gray", linestyle="--", label="unfreeze layer4")
    ax.set_xlabel("Epochs")
    ax.legend()

  plt.tight_layout()
  plt.show()



In [ ]:
plot_history(history_head, history_finetune, switch_epoch=len(history_head["train_loss"]))

In [ ]:
#Final with test data
@torch.no_grad()
def collect_predictions(model, loader):
  model.eval()
  all_preds, all_labels=[],[]

  for images, labels in loader:
    images=images.to(DEVICE)
    preds=model(images).argmax(1).cpu
    all_preds.append(preds)
    all_labels.append(labels)
  return torch.cat(all_preds), torch.cat(all_labels)
#

In [ ]:
test_preds, test_labels=collect_predictions(model, test_loader)
test_acc= (test_preds==test_labels).mean()
print(f"Test accuracy: {test_acc}")
print(classification_report(test_labels,test_preds, target_names=label_names ))

cm=confusion_matrix(test_labels, test_preds)

fig, ax= plt.subplots(5,4)
im= plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(label_names)))
ax.set_yticks(np.arange(len(label_names)))

ax.set_xticklabels(label_names)
ax.set_yticklabels(label_names)

ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
plt.colorbar(im)
plt.tight_layout()
plt.show()